# 03 — Entraînement Baseline LoRA vs Contribution (Gradients & Normales)

Ce notebook (Étape D.3) est conçu pour un seul **« Tout exécuter »** sur Google Colab (GPU T4) :
1. Monte Google Drive et sauvegarde tous les checkpoints (`last.pt`, `best.pt`), historiques (`history.json`) et métriques (`eval_test_median.json`) directement sur Drive.
2. Reprend automatiquement là où l'entraînement s'est arrêté en cas de coupure Colab (`auto_resume=True`).
3. Compare scientifiquement :
   - **Baseline LoRA** (`configs/c3vd_lora_baseline.yaml` : `L1 + SiLog` uniquement)
   - **Contribution PFE** (`configs/c3vd_lora_grad_norm.yaml` : `L1 + SiLog + Gradients multi-échelles + Normales 3D`)
4. Produit un tableau comparatif final (incluant le Zero-Shot si déjà exécuté) dans `results/comparison_summary.json`.

In [ ]:
import importlib.util
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# === PARAMÈTRES UTILISATEUR ===
REPO_URL = ""  # Optionnel sur Colab : URL GitHub de ton repo endo-depth
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/endo-depth")
DATASET_NAME = "c3vd"   # "c3vd" (recommandé : contient les normales GT) ou "simcol3d"
EXPECTED_SEQUENCES = ["cecum_t1_a", "desc_t4_a", "trans_t1_a"]  # Mettre [] pour désactiver la vérif stricte
EPOCHS = 15
BATCH_SIZE = 4
RUN_LORA_BASELINE = True
RUN_LORA_GRAD_NORM = True
AUTO_RESUME = True

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    if Path("/content/endo-depth/src").is_dir():
        PROJECT_ROOT = Path("/content/endo-depth")
        if (PROJECT_ROOT / ".git").is_dir():
            !git -C {PROJECT_ROOT} pull --ff-only || true
    elif (DRIVE_PROJECT_DIR / "src").is_dir():
        PROJECT_ROOT = DRIVE_PROJECT_DIR
    elif REPO_URL:
        !git clone {REPO_URL} /content/endo-depth
        PROJECT_ROOT = Path("/content/endo-depth")
    else:
        raise RuntimeError(
            "Dépôt endo-depth introuvable. Place-le dans MyDrive/endo-depth ou renseigne REPO_URL."
        )
    DATA_BASE = DRIVE_PROJECT_DIR / "data"
    LOCAL_DATA_BASE = Path("/content/data")
    RESULTS_DIR = DRIVE_PROJECT_DIR / "results"
else:
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
    DATA_BASE = PROJECT_ROOT / "data"
    LOCAL_DATA_BASE = PROJECT_ROOT / "data"
    RESULTS_DIR = PROJECT_ROOT / "results"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

missing = [p for p in ["transformers", "timm", "yaml", "cv2"] if importlib.util.find_spec(p) is None]
if missing:
    %pip install -q "transformers>=4.45.0" timm pyyaml opencv-python-headless

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RESULTS_DIR  :", RESULTS_DIR)

In [ ]:
import torch
from src.datasets import resolve_dataset_dir

print("CUDA dispo :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

dataset_dir, DATASET_NAME, is_demo = resolve_dataset_dir(
    DATASET_NAME,
    drive_data_base=DATA_BASE,
    local_data_base=LOCAL_DATA_BASE,
    demo_fallback_dir=RESULTS_DIR / "_demo_c3vd",
    expected_sequences=EXPECTED_SEQUENCES if EXPECTED_SEQUENCES else None,
)
if is_demo:
    print("[ATTENTION] Aucun dataset réel détecté -> mode démo synthétique C3VD activé.")
    print("Place tes fichiers dans MyDrive/endo-depth/data/c3vd_zips/ ou data/c3vd/")
    EPOCHS = min(EPOCHS, 3)
else:
    print("Dataset préparé et lu depuis :", dataset_dir)

In [ ]:
from src.train import run_training
from src.utils import load_config

def prepare_config(yaml_name: str) -> dict:
    cfg = load_config(PROJECT_ROOT / "configs" / yaml_name)
    exp_name = cfg["experiment_name"]
    cfg["data"]["dataset_name"] = DATASET_NAME
    cfg["data"]["dataset_path"] = str(dataset_dir)
    cfg["data"]["expected_sequences"] = None if (is_demo or not EXPECTED_SEQUENCES) else EXPECTED_SEQUENCES
    cfg["data"]["batch_size"] = BATCH_SIZE
    cfg["paths"]["results_dir"] = str(RESULTS_DIR)
    cfg["paths"]["checkpoint_dir"] = str(RESULTS_DIR / exp_name / "checkpoints")
    cfg["training"]["epochs"] = EPOCHS
    if DATASET_NAME == "simcol3d":
        cfg["data"]["max_depth_mm"] = 200.0
        cfg["evaluation"]["max_depth"] = 200.0
    return cfg

if RUN_LORA_BASELINE:
    print("\n=== 1/2 : Entraînement Baseline LoRA (L1 + SiLog) ===")
    cfg_base = prepare_config("c3vd_lora_baseline.yaml")
    run_training(cfg_base, auto_resume=AUTO_RESUME)

if RUN_LORA_GRAD_NORM:
    print("\n=== 2/2 : Entraînement Contribution LoRA + Gradients & Normales ===")
    cfg_contrib = prepare_config("c3vd_lora_grad_norm.yaml")
    run_training(cfg_contrib, auto_resume=AUTO_RESUME)

In [ ]:
import json
import matplotlib.pyplot as plt
from src.utils import save_json

experiments = [
    ("Zero-Shot DAV2-Small", "c3vd_zeroshot_dav2_small"),
    ("Baseline LoRA (L1+SiLog)", "c3vd_lora_baseline"),
    ("Contrib LoRA + Grad/Norm", "c3vd_lora_grad_norm"),
]

summary = {}
for label, exp_dir_name in experiments:
    eval_file = RESULTS_DIR / exp_dir_name / "eval_test_median.json"
    if eval_file.is_file():
        with eval_file.open("r", encoding="utf-8") as f:
            data = json.load(f)
        summary[label] = data["metrics"]

save_json(summary, RESULTS_DIR / "comparison_summary.json")
print("=== Tableau comparatif sur le split test ===")
print(json.dumps(summary, indent=2, ensure_ascii=False))

# Courbes de loss & validation AbsRel
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, exp_dir_name in experiments[1:]:
    hist_file = RESULTS_DIR / exp_dir_name / "history.json"
    if not hist_file.is_file():
        continue
    with hist_file.open("r", encoding="utf-8") as f:
        hist = json.load(f)
    ep = [h["epoch"] for h in hist]
    tr_loss = [h["train_loss"]["total"] for h in hist]
    va_abs = [h["val_metrics"]["abs_rel"] for h in hist]
    axes[0].plot(ep, tr_loss, marker="o", label=label)
    axes[1].plot(ep, va_abs, marker="o", label=label)

axes[0].set_title("Loss d'entraînement")
axes[0].set_xlabel("Époque")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_title("AbsRel Validation (plus bas = meilleur)")
axes[1].set_xlabel("Époque")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()